# KAC Component Ablation — SpotLight (Open RAN)

This notebook runs a clean **KAC-only** component ablation on the **same final SpotLight split** used by `KAC_Uncertainty_SpotLight_rawwidth.ipynb` and reported in Table 5 of the IEEE ICDM 2026 paper (2,708 / 580 / 582 windows; 50/50 class balance; 452 KPIs).

Four variants × five seeds = 20 training runs. Results are saved to `results/kac_ablation.csv` and aggregated to `results/kac_ablation_summary.csv`.

| Code | Variant | Text | KPI contrastive | Uncertainty $\omega$ |
| ---- | ------- | ---- | --------------- | -------------------- |
| V1   | Residual-only                          | off | off              | off |
| V2   | + Text fusion                          | on  | off              | off |
| V3   | + KPI contrastive (uniform $\omega_k$) | on  | on (uniform)     | off |
| V4   | **Full KAC** (uncertainty-weighted)    | on  | on               | on  |

Run this notebook with the working directory set to `evaluation_ver2/SpotLight` (the same directory as `KAC_Uncertainty_SpotLight_rawwidth.ipynb`).

## 1. Setup and import the shared ablation module

In [ ]:
import os, sys
from pathlib import Path

HERE = Path.cwd().resolve()
PARENT = HERE.parent
if str(PARENT) not in sys.path:
    sys.path.insert(0, str(PARENT))

assert HERE.name == "SpotLight", (
    f"Run this notebook from evaluation_ver2/SpotLight, not from {HERE}."
)
assert (HERE / "data" / "SpotLight_train.npz").exists(), (
    "data/SpotLight_train.npz not found — run KAC_Uncertainty_SpotLight_rawwidth.ipynb "
    "first to make sure the data and Chronos residual cache are in place."
)
assert (HERE / "data" / "features_cache" / "residuals_train.npy").exists(), (
    "Residual cache not found at data/features_cache/residuals_train.npy."
)

import kac_ablation
from kac_ablation import (
    DEVICE, DATASET_CONFIGS, VARIANTS, load_dataset, run_ablation,
)

print("Device:", DEVICE)
print("Loaded shared module:", kac_ablation.__file__)
print("\nVariants in this ablation:")
for v in VARIANTS:
    print(f"  {v.code}: {v.name}")

## 2. Verify we are evaluating on the *same* final split as the paper

The expected counts are **2,708 train / 580 val / 582 test**, with a 50/50 class balance and 452 KPIs (206 radio + 246 platform) sampled at 100 ms. If anything below disagrees with these numbers, **stop** and investigate — the ablation must run on the same split as Table 5 of the paper.

In [ ]:
cfg = DATASET_CONFIGS["SpotLight"]
data = load_dataset(cfg)

n_tr, n_va, n_te = len(data["y_train"]), len(data["y_val"]), len(data["y_test"])
p_tr, p_va, p_te = int(data["y_train"].sum()), int(data["y_val"].sum()), int(data["y_test"].sum())
K = data["n_kpis"]

print(f"Train : {n_tr:>4d} windows ({p_tr} positives, {p_tr/n_tr:.1%})")
print(f"Val   : {n_va:>4d} windows ({p_va} positives, {p_va/n_va:.1%})")
print(f"Test  : {n_te:>4d} windows ({p_te} positives, {p_te/n_te:.1%})")
print(f"KPIs  : {K}  (feats per KPI: {data['feats_per_kpi']})")
print(f"Residual tensor shape: {data['R_train'].shape}")

EXPECTED = {"train": 2708, "val": 580, "test": 582, "K": 452}
assert (n_tr, n_va, n_te, K) == (EXPECTED["train"], EXPECTED["val"], EXPECTED["test"], EXPECTED["K"]), (
    f"Split mismatch! Got ({n_tr},{n_va},{n_te},K={K}) but paper uses "
    f"({EXPECTED['train']},{EXPECTED['val']},{EXPECTED['test']},K={EXPECTED['K']}). "
    "Re-run KAC_Uncertainty_SpotLight_rawwidth.ipynb's data-prep cells before retrying."
)
print("\n✓ Split matches the headline KAC notebook and Table 5 of the paper.")

## 3. Run the ablation

Trains 4 variants × 5 seeds = 20 models, each up to 80 epochs with patience 15. SpotLight is the longest of the three datasets because $K{=}452$ inflates the per-KPI loss; expect ~25 min per run on A100, ~2 h per run on M-series mps. Use `seeds=[42]` and `variants=["V1", "V4"]` for a smoke test first.

In [ ]:
rows, summary_rows = run_ablation(
    dataset="SpotLight",
    seeds=[42, 123, 456, 789, 1337],
    variants=["V1", "V2", "V3", "V4"],
    out="results/kac_ablation.csv",
    epochs=80,
    patience=15,
)

## 4. View the summary table

In [ ]:
import pandas as pd

summary = pd.read_csv("results/kac_ablation_summary.csv")
fmt = lambda m, s: f"{m:.3f}±{s:.3f}"
summary["F1"]    = [fmt(m, s) for m, s in zip(summary["f1_mean"],    summary["f1_std"])]
summary["AUROC"] = [fmt(m, s) for m, s in zip(summary["auroc_mean"], summary["auroc_std"])]
summary["AP"]    = [fmt(m, s) for m, s in zip(summary["ap_mean"],    summary["ap_std"])]
summary[["variant_code", "variant_name", "n_seeds", "F1", "AUROC", "AP"]]